In [1]:
!pip install pymongo fiona geopandas --quiet


In [2]:
import fiona
import geopandas as gpd

!wget -O FPA_FOD_20221014.zip "https://www.fs.usda.gov/rds/archive/products/RDS-2013-0009.6/RDS-2013-0009.6_Data_Format3_GPKG.zip"
!unzip FPA_FOD_20221014.zip -d Data/

--2026-04-20 15:36:17--  https://www.fs.usda.gov/rds/archive/products/RDS-2013-0009.6/RDS-2013-0009.6_Data_Format3_GPKG.zip
Resolving www.fs.usda.gov (www.fs.usda.gov)... 20.140.56.69
Connecting to www.fs.usda.gov (www.fs.usda.gov)|20.140.56.69|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 221225738 (211M) [application/x-zip-compressed]
Saving to: ‘FPA_FOD_20221014.zip’

FPA_FOD_20221014.zi 100%[===================>] 210.98M  4.72MB/s    in 69s     

2026-04-20 15:37:27 (3.06 MB/s) - ‘FPA_FOD_20221014.zip’ saved [221225738/221225738]

Archive:  FPA_FOD_20221014.zip
   creating: Data//Data
  inflating: Data//Data/FPA_FOD_20221014.gpkg  
  inflating: Data//Data/_variable_descriptions.csv  


In [2]:
FOLDER_PATH = "/Users/thomascusick/Desktop/project_2/Data/Data/FPA_FOD_20221014.gpkg"

layers = fiona.listlayers(FOLDER_PATH)

fires = gpd.read_file("Data/Data/FPA_FOD_20221014.gpkg", layer="Fires")

In [3]:
import pymongo
import pandas as pd
from tqdm import tqdm

cols_to_keep = [
    'FOD_ID', 'FIRE_YEAR', 'DISCOVERY_DATE', 'DISCOVERY_DOY',
    'NWCG_CAUSE_CLASSIFICATION', 'NWCG_GENERAL_CAUSE',
    'CONT_DATE', 'CONT_DOY', 'FIRE_SIZE', 'FIRE_SIZE_CLASS',
    'LATITUDE', 'LONGITUDE', 'STATE', 'COUNTY', 'OWNER_DESCR'
]

fires_slim = fires[cols_to_keep].copy()
fires_slim['DISCOVERY_DATE'] = fires_slim['DISCOVERY_DATE'].astype(str)
fires_slim['CONT_DATE'] = fires_slim['CONT_DATE'].astype(str)

print(f"Slim shape: {fires_slim.shape}")
print(f"Estimated size: {fires_slim.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Slim shape: (2303566, 15)
Estimated size: 1143.3 MB


In [4]:
def shape_document(row):
    return {
        "fod_id": row["FOD_ID"],
        "fire_year": row["FIRE_YEAR"],
        "cause": {
            "classification": row["NWCG_CAUSE_CLASSIFICATION"],
            "general_cause": row["NWCG_GENERAL_CAUSE"]
        },
        "discovery": {
            "date": row["DISCOVERY_DATE"],
            "day_of_year": row["DISCOVERY_DOY"]
        },
        "containment": {
            "date": row["CONT_DATE"],
            "day_of_year": row["CONT_DOY"]
        },
        "size": {
            "acres": row["FIRE_SIZE"],
            "size_class": row["FIRE_SIZE_CLASS"]
        },
        "location": {
            "latitude": row["LATITUDE"],
            "longitude": row["LONGITUDE"],
            "state": row["STATE"],
            "county": row["COUNTY"],
            "owner": row["OWNER_DESCR"]
        }
    }

fires_docs = fires_slim.apply(shape_document, axis=1).tolist()
print(f"Shaped {len(fires_docs):,} documents")
print("Sample:", fires_docs[0])

Shaped 2,303,566 documents
Sample: {'fod_id': 1, 'fire_year': 2005, 'cause': {'classification': 'Human', 'general_cause': 'Power generation/transmission/distribution'}, 'discovery': {'date': '2/2/2005', 'day_of_year': 33}, 'containment': {'date': '2/2/2005', 'day_of_year': 33.0}, 'size': {'acres': 0.1, 'size_class': 'A'}, 'location': {'latitude': 40.03694444, 'longitude': -121.00583333, 'state': 'CA', 'county': '63', 'owner': 'USFS'}}


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads variables from .env file into environment

# ── Connect to MongoDB Atlas ─────────────────────────────────────────
# set Mongo URI in .env file prior to this step with proper username and password
MONGO_URI = os.getenv("MONGO_URI")  # reads URI from .env instead of hardcoding
client = pymongo.MongoClient(MONGO_URI)
db = client["ds4320"]
collection = db["hw10"]

In [6]:
collection.drop()
print("Collection cleared and ready!")

def check_storage_mb():
    stats = db.command("dbStats")
    return stats["storageSize"] / (1024 * 1024)

print(f"Storage before insert: {check_storage_mb():.1f} MB / 512 MB")

docs_to_insert = fires_docs[:300000]
collection.insert_many(docs_to_insert)

final_count = collection.count_documents({})
final_storage = check_storage_mb()

print(f"\n✅ Done!")
print(f"   Records inserted : {final_count:,}")

Collection cleared and ready!
Storage before insert: 78.8 MB / 512 MB

✅ Done!
   Records inserted : 300,000
